# 07_03C — CatBoost Nativo — Early-Stage

## Objetivo

Treinar e avaliar um **CatBoostClassifier nativo** para prever risco de perda jurídica do banco usando a ABT early-stage criada nas etapas anteriores.

A motivação desta etapa é testar um modelo tabular mais adequado para o seu problema, porque o CatBoost:

- lida nativamente com variáveis categóricas;
- captura interações entre categorias sem precisar de one-hot grande;
- funciona bem com alta cardinalidade;
- permite `auto_class_weights` ou `class_weights`;
- pode usar variáveis textuais de forma nativa, se desejado;
- costuma ser forte em bases tabulares com muitos campos cadastrais e históricos.

## Target

Neste projeto, a semântica correta é:

```text
target_banco_ganhou = 0 → banco ganhou / êxito
target_banco_ganhou = 1 → banco perdeu / condenação
```

Logo, neste notebook:

```text
classe positiva = 1 = banco perdeu
proba_perda = predict_proba(...)[1]
score_risco_perda = proba_perda
score_prioridade_financeira = proba_perda × fe_valor_ajuizado
```

## Entradas esperadas

```text
data/processed/abt_early_stage_v1_dev_hist.parquet
data/processed/abt_early_stage_v1_holdout_hist.parquet
data/processed/feature_list_early_stage_v1_hist.txt
```

## Baseline campeão atual

O modelo campeão atual vem da etapa `07_03A`:

```text
Random Forest + OneHot + TF-IDF
PR-AUC perda holdout ≈ 0,4668
ROC-AUC perda holdout ≈ 0,7786
Top 5% risco perda ≈ 60,30%
Top 10% financeiro ≈ 50,26% do valor das perdas
```

O CatBoost nativo só deve substituir o baseline se melhorar uma dessas frentes sem piorar muito as demais.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import importlib.util

warnings.filterwarnings("ignore")

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

pd.set_option("display.max_columns", 400)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 260)

BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "outputs" / "reports" / "modelagem_early_stage"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DEV_HIST_FILE = PROCESSED_DIR / "abt_early_stage_v1_dev_hist.parquet"
HOLDOUT_HIST_FILE = PROCESSED_DIR / "abt_early_stage_v1_holdout_hist.parquet"
FEATURE_LIST_HIST_FILE = PROCESSED_DIR / "feature_list_early_stage_v1_hist.txt"

TARGET_COL_LEGACY = "target_banco_ganhou"
TARGET_COL = "target_banco_perdeu"

DATE_COL = "Datainicial"
VALUE_COL = "fe_valor_ajuizado"

RANDOM_STATE = 42

print("Setup concluído.")
print("Target semântico: 0=banco_ganhou | 1=banco_perdeu")

## 2. Checar dependência CatBoost

In [ ]:
if importlib.util.find_spec("catboost") is None:
    raise ImportError("CatBoost não está instalado. Rode: pip install catboost")

from catboost import CatBoostClassifier, Pool
print("CatBoost importado com sucesso.")

## 3. Carregar bases

In [ ]:
df_dev = pd.read_parquet(DEV_HIST_FILE)
df_holdout = pd.read_parquet(HOLDOUT_HIST_FILE)

with open(FEATURE_LIST_HIST_FILE, "r", encoding="utf-8") as f:
    feature_list = [line.strip() for line in f if line.strip()]

df_dev[DATE_COL] = pd.to_datetime(df_dev[DATE_COL], errors="coerce")
df_holdout[DATE_COL] = pd.to_datetime(df_holdout[DATE_COL], errors="coerce")

# Criar target semântico correto.
df_dev[TARGET_COL] = df_dev[TARGET_COL_LEGACY].astype(int)
df_holdout[TARGET_COL] = df_holdout[TARGET_COL_LEGACY].astype(int)

feature_list = [c for c in feature_list if c in df_dev.columns and c in df_holdout.columns]

print("Dev:", df_dev.shape)
print("Holdout:", df_holdout.shape)
print("Features disponíveis:", len(feature_list))
display(df_dev.head())

## 4. Funções auxiliares

In [ ]:
def save_report(df_report, filename):
    path = REPORTS_DIR / filename
    df_report.to_csv(path, index=False)
    print(f"Relatório salvo em: {path}")


def target_distribution(df, name):
    out = df[TARGET_COL].value_counts(dropna=False).rename_axis("target").reset_index(name="qtd")
    out["classe"] = out["target"].map({0: "banco_ganhou", 1: "banco_perdeu"})
    out["perc"] = out["qtd"] / out["qtd"].sum()
    out["dataset"] = name
    return out


def infer_feature_types(df, features):
    numeric_features = []
    categorical_features = []
    text_features = []

    for col in features:
        if col == "fe_texto_inicial_curto":
            text_features.append(col)
        elif pd.api.types.is_numeric_dtype(df[col]):
            numeric_features.append(col)
        else:
            categorical_features.append(col)

    return numeric_features, categorical_features, text_features


def clean_catboost_df(df, features, cat_features, text_features=None):
    text_features = text_features or []
    X = df[features].copy()

    for col in features:
        if col in cat_features or col in text_features:
            X[col] = X[col].fillna("nao_informado").astype(str)
        else:
            X[col] = pd.to_numeric(X[col], errors="coerce")

    return X


def make_temporal_folds(df, date_col=DATE_COL):
    folds = [(0.55, 0.70), (0.70, 0.85), (0.85, 1.00)]
    df_sorted = df.sort_values(date_col).copy()
    dates = df_sorted[date_col]
    split_rows = []

    for i, (train_end_q, val_end_q) in enumerate(folds, start=1):
        train_end_date = dates.quantile(train_end_q)
        val_end_date = dates.quantile(val_end_q)

        train_idx = df_sorted.index[df_sorted[date_col] <= train_end_date]
        val_idx = df_sorted.index[(df_sorted[date_col] > train_end_date) & (df_sorted[date_col] <= val_end_date)]

        split_rows.append({
            "fold": i,
            "train_start_date": df_sorted.loc[train_idx, date_col].min(),
            "train_end_date": train_end_date,
            "val_start_date": df_sorted.loc[val_idx, date_col].min(),
            "val_end_date": val_end_date,
            "train_idx": train_idx,
            "val_idx": val_idx,
            "qtd_train": len(train_idx),
            "qtd_val": len(val_idx),
        })

    return split_rows


def threshold_metrics_perda(y_true, proba_perda, threshold=0.5):
    pred = (proba_perda >= threshold).astype(int)
    return {
        "threshold": threshold,
        "precision_perda": precision_score(y_true, pred, zero_division=0),
        "recall_perda": recall_score(y_true, pred, zero_division=0),
        "f1_perda": f1_score(y_true, pred, zero_division=0),
        "pred_perda_rate": pred.mean(),
    }


def find_best_f1_threshold_perda(y_true, proba_perda):
    precision, recall, thresholds = precision_recall_curve(y_true, proba_perda)

    if len(thresholds) == 0:
        return 0.5, 0, 0, 0

    precision_t = precision[:-1]
    recall_t = recall[:-1]
    f1_t = 2 * precision_t * recall_t / np.maximum(precision_t + recall_t, 1e-12)
    best_idx = np.nanargmax(f1_t)

    return thresholds[best_idx], precision_t[best_idx], recall_t[best_idx], f1_t[best_idx]


def topk_risco_perda_metrics(y_true, proba_perda, ks=(0.01, 0.05, 0.10, 0.20)):
    df_tmp = pd.DataFrame({"y_true": y_true, "score_risco_perda": proba_perda})
    df_tmp = df_tmp.sort_values("score_risco_perda", ascending=False).reset_index(drop=True)

    total_perdas = (df_tmp["y_true"] == 1).sum()
    taxa_perda_base = (df_tmp["y_true"] == 1).mean()

    out = []
    for k in ks:
        n = max(1, int(np.ceil(len(df_tmp) * k)))
        top = df_tmp.head(n)
        precision_perda_at_k = (top["y_true"] == 1).mean()
        recall_perda_at_k = (top["y_true"] == 1).sum() / total_perdas if total_perdas else np.nan
        lift_perda_at_k = precision_perda_at_k / taxa_perda_base if taxa_perda_base else np.nan

        out.append({
            "top_k": k,
            "n_top": n,
            "precision_perda_at_k": precision_perda_at_k,
            "recall_perda_at_k": recall_perda_at_k,
            "lift_perda_at_k": lift_perda_at_k,
            "taxa_perda_base": taxa_perda_base,
        })

    return pd.DataFrame(out)


def topk_prioridade_financeira_metrics(y_true, proba_perda, valor_ajuizado, ks=(0.01, 0.05, 0.10, 0.20)):
    df_tmp = pd.DataFrame({
        "y_true": y_true,
        "proba_perda": proba_perda,
        "valor_ajuizado": pd.to_numeric(valor_ajuizado, errors="coerce").fillna(0),
    })

    df_tmp["score_prioridade_financeira"] = df_tmp["proba_perda"] * df_tmp["valor_ajuizado"]
    df_tmp = df_tmp.sort_values("score_prioridade_financeira", ascending=False).reset_index(drop=True)

    total_perdas = (df_tmp["y_true"] == 1).sum()
    taxa_perda_base = (df_tmp["y_true"] == 1).mean()
    total_valor = df_tmp["valor_ajuizado"].sum()
    total_valor_perdas = df_tmp.loc[df_tmp["y_true"] == 1, "valor_ajuizado"].sum()

    out = []
    for k in ks:
        n = max(1, int(np.ceil(len(df_tmp) * k)))
        top = df_tmp.head(n)
        valor_top = top["valor_ajuizado"].sum()
        valor_perdas_top = top.loc[top["y_true"] == 1, "valor_ajuizado"].sum()
        precision_perda_at_k = (top["y_true"] == 1).mean()
        recall_perda_at_k = (top["y_true"] == 1).sum() / total_perdas if total_perdas else np.nan
        lift_perda_at_k = precision_perda_at_k / taxa_perda_base if taxa_perda_base else np.nan

        out.append({
            "top_k": k,
            "n_top": n,
            "precision_perda_at_k": precision_perda_at_k,
            "recall_perda_at_k": recall_perda_at_k,
            "lift_perda_at_k": lift_perda_at_k,
            "taxa_perda_base": taxa_perda_base,
            "valor_ajuizado_top": valor_top,
            "share_valor_ajuizado_total": valor_top / total_valor if total_valor else np.nan,
            "valor_ajuizado_perdas_top": valor_perdas_top,
            "share_valor_perdas_total": valor_perdas_top / total_valor_perdas if total_valor_perdas else np.nan,
        })

    return pd.DataFrame(out)

## 5. Distribuição do target

In [ ]:
target_dist = pd.concat([
    target_distribution(df_dev, "dev"),
    target_distribution(df_holdout, "holdout"),
], ignore_index=True)

save_report(target_dist, "49_3c_target_distribution.csv")
target_dist

## 6. Separar features por tipo

In [ ]:
numeric_features, categorical_features, text_features = infer_feature_types(df_dev, feature_list)

feature_type_summary = pd.DataFrame([
    {"tipo": "numeric", "qtd": len(numeric_features)},
    {"tipo": "categorical", "qtd": len(categorical_features)},
    {"tipo": "text", "qtd": len(text_features)},
])

save_report(feature_type_summary, "50_3c_feature_type_summary.csv")

print("Numéricas:", len(numeric_features))
print("Categóricas:", len(categorical_features))
print("Texto:", len(text_features))
feature_type_summary

## 7. Opção de uso de texto

Por padrão, este notebook começa com `USE_TEXT_FEATURES = False`.

Motivo: CatBoost com texto pode funcionar bem, mas costuma exigir mais tempo e pode variar de ambiente para ambiente.

Estratégia recomendada:

1. Primeiro rode sem texto.
2. Se rodar rápido e estável, rode novamente com `USE_TEXT_FEATURES = True`.

O texto esperado é:

```text
fe_texto_inicial_curto
```

In [ ]:
USE_TEXT_FEATURES = False

if not USE_TEXT_FEATURES:
    selected_text_features = []
    selected_features = [c for c in feature_list if c not in text_features]
else:
    selected_text_features = text_features
    selected_features = feature_list.copy()

selected_numeric_features = [c for c in numeric_features if c in selected_features]
selected_categorical_features = [c for c in categorical_features if c in selected_features]

cat_features_idx = [selected_features.index(c) for c in selected_categorical_features]
text_features_idx = [selected_features.index(c) for c in selected_text_features]

print("USE_TEXT_FEATURES:", USE_TEXT_FEATURES)
print("Features selecionadas:", len(selected_features))
print("Cat features:", len(selected_categorical_features))
print("Text features:", len(selected_text_features))
print("Numeric features:", len(selected_numeric_features))

## 8. Criar folds temporais

In [ ]:
folds = make_temporal_folds(df_dev, DATE_COL)

fold_summary_rows = []
for fold in folds:
    train_idx = fold["train_idx"]
    val_idx = fold["val_idx"]
    taxa_perda_train = df_dev.loc[train_idx, TARGET_COL].mean()
    taxa_perda_val = df_dev.loc[val_idx, TARGET_COL].mean()

    fold_summary_rows.append({
        "fold": fold["fold"],
        "train_start_date": fold["train_start_date"],
        "train_end_date": fold["train_end_date"],
        "val_start_date": fold["val_start_date"],
        "val_end_date": fold["val_end_date"],
        "qtd_train": fold["qtd_train"],
        "qtd_val": fold["qtd_val"],
        "taxa_perda_train": taxa_perda_train,
        "taxa_perda_val": taxa_perda_val,
        "taxa_ganho_train": 1 - taxa_perda_train,
        "taxa_ganho_val": 1 - taxa_perda_val,
    })

fold_summary = pd.DataFrame(fold_summary_rows)
save_report(fold_summary, "51_3c_walk_forward_folds_summary.csv")
fold_summary

## 9. Definir candidatos CatBoost

Vamos testar alguns perfis:

1. `catboost_balanced_default`
2. `catboost_balanced_depth6`
3. `catboost_balanced_depth8`
4. `catboost_lossguide`

Se ficar lento, reduza `iterations` para 400.

In [ ]:
def make_catboost_model(config_name):
    common = dict(
        loss_function="Logloss",
        eval_metric="PRAUC",
        custom_metric=["AUC", "Precision", "Recall", "F1"],
        auto_class_weights="Balanced",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )

    configs = {
        "catboost_balanced_default": dict(
            iterations=700,
            learning_rate=0.04,
            depth=6,
            l2_leaf_reg=8,
            random_strength=1.0,
            bagging_temperature=0.5,
            early_stopping_rounds=60,
        ),
        "catboost_balanced_depth6": dict(
            iterations=900,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=12,
            random_strength=1.5,
            bagging_temperature=0.8,
            early_stopping_rounds=80,
        ),
        "catboost_balanced_depth8": dict(
            iterations=700,
            learning_rate=0.03,
            depth=8,
            l2_leaf_reg=10,
            random_strength=1.0,
            bagging_temperature=0.8,
            early_stopping_rounds=70,
        ),
        "catboost_lossguide": dict(
            iterations=700,
            learning_rate=0.04,
            grow_policy="Lossguide",
            max_leaves=31,
            depth=8,
            l2_leaf_reg=12,
            random_strength=1.0,
            bagging_temperature=0.7,
            early_stopping_rounds=70,
        ),
    }

    params = common.copy()
    params.update(configs[config_name])
    return CatBoostClassifier(**params)


RUN_MODE = "fast"  # "fast" ou "full"

if RUN_MODE == "fast":
    catboost_candidates = ["catboost_balanced_default", "catboost_balanced_depth6"]
else:
    catboost_candidates = [
        "catboost_balanced_default",
        "catboost_balanced_depth6",
        "catboost_balanced_depth8",
        "catboost_lossguide",
    ]

print("Candidatos:")
for c in catboost_candidates:
    print(" -", c)

## 10. Rodar walk-forward CatBoost

In [ ]:
def evaluate_catboost_on_fold(model_name, df, train_idx, val_idx):
    X_train = clean_catboost_df(df.loc[train_idx], selected_features, selected_categorical_features, selected_text_features)
    y_train = df.loc[train_idx, TARGET_COL].astype(int)

    X_val = clean_catboost_df(df.loc[val_idx], selected_features, selected_categorical_features, selected_text_features)
    y_val = df.loc[val_idx, TARGET_COL].astype(int)

    train_pool = Pool(
        X_train,
        y_train,
        cat_features=cat_features_idx,
        text_features=text_features_idx if selected_text_features else None,
        feature_names=selected_features,
    )

    val_pool = Pool(
        X_val,
        y_val,
        cat_features=cat_features_idx,
        text_features=text_features_idx if selected_text_features else None,
        feature_names=selected_features,
    )

    model = make_catboost_model(model_name)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    proba_perda_val = model.predict_proba(val_pool)[:, 1]

    roc_auc_perda = roc_auc_score(y_val, proba_perda_val)
    pr_auc_perda = average_precision_score(y_val, proba_perda_val)

    best_thr, best_p, best_r, best_f1 = find_best_f1_threshold_perda(y_val, proba_perda_val)
    thr_05 = threshold_metrics_perda(y_val, proba_perda_val, threshold=0.5)

    result = {
        "model": model_name,
        "roc_auc_perda": roc_auc_perda,
        "pr_auc_perda": pr_auc_perda,
        "taxa_perda_val": y_val.mean(),
        "taxa_ganho_val": 1 - y_val.mean(),
        "best_f1_threshold_perda": best_thr,
        "best_f1_precision_perda": best_p,
        "best_f1_recall_perda": best_r,
        "best_f1_perda": best_f1,
        "precision_perda_t05": thr_05["precision_perda"],
        "recall_perda_t05": thr_05["recall_perda"],
        "f1_perda_t05": thr_05["f1_perda"],
        "pred_perda_rate_t05": thr_05["pred_perda_rate"],
        "best_iteration": model.get_best_iteration(),
    }

    topk_perda = topk_risco_perda_metrics(y_val.values, proba_perda_val)

    if VALUE_COL in df.columns:
        topk_financeiro = topk_prioridade_financeira_metrics(
            y_true=y_val.values,
            proba_perda=proba_perda_val,
            valor_ajuizado=df.loc[val_idx, VALUE_COL].values,
        )
    else:
        topk_financeiro = pd.DataFrame()

    return result, topk_perda, topk_financeiro


walk_forward_results = []
topk_perda_results = []
topk_financeiro_results = []

for model_name in catboost_candidates:
    print("=" * 100)
    print(f"Modelo: {model_name}")

    for fold in folds:
        print(f"  Fold {fold['fold']}")
        result, topk_perda, topk_financeiro = evaluate_catboost_on_fold(
            model_name=model_name,
            df=df_dev,
            train_idx=fold["train_idx"],
            val_idx=fold["val_idx"],
        )

        result["fold"] = fold["fold"]
        result["qtd_train"] = fold["qtd_train"]
        result["qtd_val"] = fold["qtd_val"]
        result["train_end_date"] = fold["train_end_date"]
        result["val_start_date"] = fold["val_start_date"]
        result["val_end_date"] = fold["val_end_date"]
        walk_forward_results.append(result)

        topk_perda["model"] = model_name
        topk_perda["fold"] = fold["fold"]
        topk_perda_results.append(topk_perda)

        if not topk_financeiro.empty:
            topk_financeiro["model"] = model_name
            topk_financeiro["fold"] = fold["fold"]
            topk_financeiro_results.append(topk_financeiro)

walk_forward_results = pd.DataFrame(walk_forward_results)
topk_perda_results = pd.concat(topk_perda_results, ignore_index=True)
topk_financeiro_results = pd.concat(topk_financeiro_results, ignore_index=True) if topk_financeiro_results else pd.DataFrame()

save_report(walk_forward_results, "52_3c_walk_forward_metrics.csv")
save_report(topk_perda_results, "53_3c_walk_forward_topk_risco_perda.csv")

if not topk_financeiro_results.empty:
    save_report(topk_financeiro_results, "54_3c_walk_forward_topk_prioridade_financeira.csv")

walk_forward_results.sort_values(["pr_auc_perda", "roc_auc_perda"], ascending=False)

## 11. Resumo por modelo

In [ ]:
model_summary = (
    walk_forward_results
    .groupby("model")
    .agg(
        mean_roc_auc_perda=("roc_auc_perda", "mean"),
        std_roc_auc_perda=("roc_auc_perda", "std"),
        mean_pr_auc_perda=("pr_auc_perda", "mean"),
        std_pr_auc_perda=("pr_auc_perda", "std"),
        mean_taxa_perda_val=("taxa_perda_val", "mean"),
        mean_best_f1_perda=("best_f1_perda", "mean"),
        mean_best_f1_threshold_perda=("best_f1_threshold_perda", "mean"),
        mean_precision_perda_t05=("precision_perda_t05", "mean"),
        mean_recall_perda_t05=("recall_perda_t05", "mean"),
        mean_f1_perda_t05=("f1_perda_t05", "mean"),
        mean_best_iteration=("best_iteration", "mean"),
    )
    .reset_index()
    .sort_values("mean_pr_auc_perda", ascending=False)
)

save_report(model_summary, "55_3c_model_summary.csv")
model_summary

## 12. Top-k risco de perda por modelo

In [ ]:
topk_perda_summary = (
    topk_perda_results
    .groupby(["model", "top_k"])
    .agg(
        mean_precision_perda_at_k=("precision_perda_at_k", "mean"),
        mean_recall_perda_at_k=("recall_perda_at_k", "mean"),
        mean_lift_perda_at_k=("lift_perda_at_k", "mean"),
        mean_taxa_perda_base=("taxa_perda_base", "mean"),
    )
    .reset_index()
    .sort_values(["top_k", "mean_precision_perda_at_k"], ascending=[True, False])
)

save_report(topk_perda_summary, "56_3c_topk_risco_perda_summary.csv")
topk_perda_summary

## 13. Top-k prioridade financeira por modelo

In [ ]:
if not topk_financeiro_results.empty:
    topk_financeiro_summary = (
        topk_financeiro_results
        .groupby(["model", "top_k"])
        .agg(
            mean_precision_perda_at_k=("precision_perda_at_k", "mean"),
            mean_recall_perda_at_k=("recall_perda_at_k", "mean"),
            mean_lift_perda_at_k=("lift_perda_at_k", "mean"),
            mean_share_valor_ajuizado_total=("share_valor_ajuizado_total", "mean"),
            mean_share_valor_perdas_total=("share_valor_perdas_total", "mean"),
        )
        .reset_index()
        .sort_values(["top_k", "mean_share_valor_perdas_total"], ascending=[True, False])
    )

    save_report(topk_financeiro_summary, "57_3c_topk_prioridade_financeira_summary.csv")
    display(topk_financeiro_summary)
else:
    print("Top-k financeiro não calculado.")

## 14. Selecionar melhor CatBoost por PR-AUC de perda

In [ ]:
best_model_name = model_summary.iloc[0]["model"]
print("Melhor CatBoost por PR-AUC de perda:", best_model_name)

## 15. Treinar melhor CatBoost no Dev completo e avaliar Holdout

In [ ]:
X_holdout = clean_catboost_df(df_holdout, selected_features, selected_categorical_features, selected_text_features)
y_holdout = df_holdout[TARGET_COL].astype(int)

# Separação interna para early stopping dentro do dev completo:
# usa os 15% mais recentes do dev como eval_set.
df_dev_ordered = df_dev.sort_values(DATE_COL).copy()
cut_eval = df_dev_ordered[DATE_COL].quantile(0.85)

train_idx_final = df_dev_ordered.index[df_dev_ordered[DATE_COL] <= cut_eval]
eval_idx_final = df_dev_ordered.index[df_dev_ordered[DATE_COL] > cut_eval]

X_train_final = clean_catboost_df(df_dev.loc[train_idx_final], selected_features, selected_categorical_features, selected_text_features)
y_train_final = df_dev.loc[train_idx_final, TARGET_COL].astype(int)

X_eval_final = clean_catboost_df(df_dev.loc[eval_idx_final], selected_features, selected_categorical_features, selected_text_features)
y_eval_final = df_dev.loc[eval_idx_final, TARGET_COL].astype(int)

train_pool_final = Pool(
    X_train_final,
    y_train_final,
    cat_features=cat_features_idx,
    text_features=text_features_idx if selected_text_features else None,
    feature_names=selected_features,
)

eval_pool_final = Pool(
    X_eval_final,
    y_eval_final,
    cat_features=cat_features_idx,
    text_features=text_features_idx if selected_text_features else None,
    feature_names=selected_features,
)

holdout_pool = Pool(
    X_holdout,
    y_holdout,
    cat_features=cat_features_idx,
    text_features=text_features_idx if selected_text_features else None,
    feature_names=selected_features,
)

best_model = make_catboost_model(best_model_name)
best_model.fit(train_pool_final, eval_set=eval_pool_final, use_best_model=True)

proba_perda_holdout = best_model.predict_proba(holdout_pool)[:, 1]

holdout_roc_auc_perda = roc_auc_score(y_holdout, proba_perda_holdout)
holdout_pr_auc_perda = average_precision_score(y_holdout, proba_perda_holdout)

best_thr, best_p, best_r, best_f1 = find_best_f1_threshold_perda(y_holdout, proba_perda_holdout)
thr_05 = threshold_metrics_perda(y_holdout, proba_perda_holdout, threshold=0.5)

holdout_metrics = pd.DataFrame([{
    "model": best_model_name,
    "roc_auc_perda": holdout_roc_auc_perda,
    "pr_auc_perda": holdout_pr_auc_perda,
    "taxa_perda_holdout": y_holdout.mean(),
    "taxa_ganho_holdout": 1 - y_holdout.mean(),
    "best_f1_threshold_perda": best_thr,
    "best_f1_precision_perda": best_p,
    "best_f1_recall_perda": best_r,
    "best_f1_perda": best_f1,
    "precision_perda_t05": thr_05["precision_perda"],
    "recall_perda_t05": thr_05["recall_perda"],
    "f1_perda_t05": thr_05["f1_perda"],
    "pred_perda_rate_t05": thr_05["pred_perda_rate"],
    "qtd_dev": len(df_dev),
    "qtd_train_final": len(train_idx_final),
    "qtd_eval_final": len(eval_idx_final),
    "qtd_holdout": len(df_holdout),
    "qtd_features": len(selected_features),
    "use_text_features": USE_TEXT_FEATURES,
    "best_iteration": best_model.get_best_iteration(),
}])

save_report(holdout_metrics, "58_3c_holdout_best_catboost_metrics.csv")
holdout_metrics

## 16. Observação importante sobre treino final

Neste notebook, por segurança, o modelo final usa:

```text
85% mais antigo do dev → treino
15% mais recente do dev → eval_set para early stopping
holdout → avaliação final
```

Isso evita usar o holdout para early stopping.

Depois, quando o modelo for escolhido oficialmente, podemos:

1. fixar o número ótimo de iterações;
2. retreinar em todo o dev;
3. avaliar em um novo holdout ou janela OOT posterior, se existir.

## 17. Holdout top-k risco de perda

In [ ]:
holdout_topk_perda = topk_risco_perda_metrics(
    y_true=y_holdout.values,
    proba_perda=proba_perda_holdout,
    ks=(0.01, 0.05, 0.10, 0.20)
)

holdout_topk_perda["model"] = best_model_name
save_report(holdout_topk_perda, "59_3c_holdout_topk_risco_perda.csv")
holdout_topk_perda

## 18. Holdout top-k prioridade financeira

In [ ]:
if VALUE_COL in df_holdout.columns:
    holdout_topk_financeiro = topk_prioridade_financeira_metrics(
        y_true=y_holdout.values,
        proba_perda=proba_perda_holdout,
        valor_ajuizado=df_holdout[VALUE_COL],
        ks=(0.01, 0.05, 0.10, 0.20)
    )

    holdout_topk_financeiro["model"] = best_model_name
    save_report(holdout_topk_financeiro, "60_3c_holdout_topk_prioridade_financeira.csv")
    display(holdout_topk_financeiro)
else:
    print(f"{VALUE_COL} não encontrado.")

## 19. Ranking do holdout

In [ ]:
ranking_cols = ["Numerodistribuicao", "Identificador", DATE_COL, TARGET_COL_LEGACY, TARGET_COL, VALUE_COL]
ranking_cols = [c for c in ranking_cols if c in df_holdout.columns]

ranking_holdout = df_holdout[ranking_cols].copy()
ranking_holdout["target_classe_real"] = ranking_holdout[TARGET_COL].map({0: "banco_ganhou", 1: "banco_perdeu"})
ranking_holdout["proba_banco_perdeu"] = proba_perda_holdout
ranking_holdout["score_risco_perda"] = proba_perda_holdout

if VALUE_COL in ranking_holdout.columns:
    ranking_holdout["score_prioridade_financeira"] = ranking_holdout["score_risco_perda"] * pd.to_numeric(ranking_holdout[VALUE_COL], errors="coerce").fillna(0)
else:
    ranking_holdout["score_prioridade_financeira"] = np.nan

ranking_holdout["rank_risco_perda"] = ranking_holdout["score_risco_perda"].rank(ascending=False, method="first").astype(int)
ranking_holdout["rank_prioridade_financeira"] = ranking_holdout["score_prioridade_financeira"].rank(ascending=False, method="first").astype(int)

ranking_risco = ranking_holdout.sort_values("score_risco_perda", ascending=False)
ranking_financeiro = ranking_holdout.sort_values("score_prioridade_financeira", ascending=False)

ranking_risco_path = PROCESSED_DIR / "ranking_holdout_risco_perda_3c_catboost.parquet"
ranking_financeiro_path = PROCESSED_DIR / "ranking_holdout_prioridade_financeira_3c_catboost.parquet"

ranking_risco.to_parquet(ranking_risco_path, index=False)
ranking_financeiro.to_parquet(ranking_financeiro_path, index=False)

save_report(ranking_risco.head(1000), "61_3c_ranking_holdout_top1000_risco_perda.csv")
save_report(ranking_financeiro.head(1000), "62_3c_ranking_holdout_top1000_prioridade_financeira.csv")

ranking_risco.head(20)

## 20. Classification report

In [ ]:
pred_perda_holdout_t05 = (proba_perda_holdout >= 0.5).astype(int)

print("Classification report — threshold 0.5")
print("Classe 0 = banco ganhou")
print("Classe 1 = banco perdeu")
print(classification_report(
    y_holdout,
    pred_perda_holdout_t05,
    target_names=["banco_ganhou", "banco_perdeu"]
))

print("Confusion matrix — threshold 0.5")
print("Linhas = real | Colunas = previsto")
print(confusion_matrix(y_holdout, pred_perda_holdout_t05))

## 21. Feature importance

In [ ]:
feature_importance = pd.DataFrame({
    "feature": selected_features,
    "importance": best_model.get_feature_importance()
}).sort_values("importance", ascending=False)

save_report(feature_importance, "63_3c_catboost_feature_importance.csv")
feature_importance.head(50)

## 22. Comparação 3A vs 3B vs 3C

In [ ]:
# Valores de referência das etapas anteriores.
# Ajuste manualmente caso seus CSVs tenham valores diferentes.

baseline_3a = {
    "model": "3A_random_forest_onehot_tfidf",
    "holdout_pr_auc_perda": 0.466804,
    "holdout_roc_auc_perda": 0.77858,
    "top5_precision_perda": 0.603004,
    "top10_fin_share_valor_perdas": 0.502560,
}

baseline_3b = {
    "model": "3B_hgb_catboost_encoder",
    "holdout_pr_auc_perda": 0.434753,
    "holdout_roc_auc_perda": 0.769694,
    "top5_precision_perda": 0.525751,
    "top10_fin_share_valor_perdas": 0.509398,
}

current_top5 = holdout_topk_perda.loc[holdout_topk_perda["top_k"] == 0.05, "precision_perda_at_k"].iloc[0]

if VALUE_COL in df_holdout.columns:
    current_top10_fin = holdout_topk_financeiro.loc[holdout_topk_financeiro["top_k"] == 0.10, "share_valor_perdas_total"].iloc[0]
else:
    current_top10_fin = np.nan

comparison_3a_3b_3c = pd.DataFrame([
    baseline_3a,
    baseline_3b,
    {
        "model": f"3C_{best_model_name}",
        "holdout_pr_auc_perda": holdout_pr_auc_perda,
        "holdout_roc_auc_perda": holdout_roc_auc_perda,
        "top5_precision_perda": current_top5,
        "top10_fin_share_valor_perdas": current_top10_fin,
    }
])

comparison_3a_3b_3c["delta_pr_auc_vs_3a"] = comparison_3a_3b_3c["holdout_pr_auc_perda"] - comparison_3a_3b_3c.loc[0, "holdout_pr_auc_perda"]
comparison_3a_3b_3c["delta_roc_auc_vs_3a"] = comparison_3a_3b_3c["holdout_roc_auc_perda"] - comparison_3a_3b_3c.loc[0, "holdout_roc_auc_perda"]
comparison_3a_3b_3c["delta_top5_precision_vs_3a"] = comparison_3a_3b_3c["top5_precision_perda"] - comparison_3a_3b_3c.loc[0, "top5_precision_perda"]
comparison_3a_3b_3c["delta_top10_fin_vs_3a"] = comparison_3a_3b_3c["top10_fin_share_valor_perdas"] - comparison_3a_3b_3c.loc[0, "top10_fin_share_valor_perdas"]

save_report(comparison_3a_3b_3c, "64_3c_comparison_3a_3b_3c.csv")
comparison_3a_3b_3c

## 23. Summary final

In [ ]:
summary_step_3c = pd.DataFrame([{
    "target_semantica": "0=banco_ganhou | 1=banco_perdeu",
    "run_mode": RUN_MODE,
    "use_text_features": USE_TEXT_FEATURES,
    "best_model_3c_walk_forward": best_model_name,
    "best_model_3c_mean_pr_auc_perda_wf": model_summary.loc[model_summary["model"] == best_model_name, "mean_pr_auc_perda"].iloc[0],
    "best_model_3c_mean_roc_auc_perda_wf": model_summary.loc[model_summary["model"] == best_model_name, "mean_roc_auc_perda"].iloc[0],
    "holdout_pr_auc_perda": holdout_pr_auc_perda,
    "holdout_roc_auc_perda": holdout_roc_auc_perda,
    "taxa_perda_holdout": y_holdout.mean(),
    "taxa_ganho_holdout": 1 - y_holdout.mean(),
    "qtd_features": len(selected_features),
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "best_iteration": best_model.get_best_iteration(),
    "ranking_risco_holdout_path": str(ranking_risco_path),
    "ranking_financeiro_holdout_path": str(ranking_financeiro_path),
}])

save_report(summary_step_3c, "65_summary_step_3c_catboost.csv")
summary_step_3c.T

# O que verificar após rodar

Envie na próxima iteração:

1. `model_summary`
2. `holdout_metrics`
3. `holdout_topk_perda`
4. `holdout_topk_financeiro`
5. `feature_importance.head(30)`
6. `comparison_3a_3b_3c`
7. `summary_step_3c.T`

## Como decidir

O CatBoost 3C deve virar campeão se superar o 3A em pelo menos um dos critérios principais:

```text
PR-AUC perda holdout
Top 5% risco perda
Top 10% financeiro
```

Sem degradar muito os outros.

## Próxima etapa provável

Depois de avaliar 3C, temos dois caminhos:

1. Se 3C ganhar:
   - seguir com 3C para seleção final e calibração.

2. Se 3C não ganhar:
   - manter 3A como campeão atual.

Em ambos os casos, a próxima etapa lógica será:

```text
07_04_model_selection_target_banco_perdeu.ipynb
```

Depois:

```text
07_05_calibracao_venn_abers_target_banco_perdeu.ipynb
```